In [ ]:
pip install -U peft

In [ ]:
!pip install -q transformers peft datasets accelerate evaluate
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import PeftModel, LoraConfig
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from datasets import Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- Paths & knobs ----
BASE_MODEL = "roberta-base"                     
DAPT_ADAPTER_DIR = "/content/drive/MyDrive/roberta/roberta-dapt-lyrics-lora"    # folder with adapter_config.json + adapter_model.bin
OUT_DIR = "/content/drive/MyDrive/roberta/multi-class-classifier"
NUM_LABELS = 6
NEW_ADAPTER_NAME = "tox_cls"
SEED = 42
MAX_LEN = 256
EPOCH = 10
torch.manual_seed(SEED)

In [ ]:
jigsaw_dir = "/content/drive/My Drive/Jigsaw/train.csv"
df = pd.read_csv(jigsaw_dir)
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
df["labels"] = df[label_cols].astype(float).values.tolist()
df.rename(columns={"comment_text": "text"}, inplace=True)
df = df[['text','labels']]
df.head()

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel, LoraConfig, get_peft_model

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=NUM_LABELS)

# --- Load old (frozen) DAPT LoRA and bake it into weights ---
model = PeftModel.from_pretrained(
    model,
    DAPT_ADAPTER_DIR,
    adapter_name="dapt",
    is_trainable=False,    # keep old adapter frozen
)
merged = model.merge_and_unload()      # old LoRA is now baked/frozen into weights

# --- Add a NEW trainable LoRA on top of the fused model ---
new_lora_cfg = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["query", "key", "value"],   # BERT/RoBERTa naming
    task_type="SEQ_CLS",
    adapter_name=NEW_ADAPTER_NAME
)

model_cls = get_peft_model(merged, new_lora_cfg)  # only NEW adapter exists now

# --- Re-enable classifier head training; base stays frozen; LoRA is trainable ---
for n, p in model_cls.named_parameters():
    if "classifier" in n:
        p.requires_grad = True

# (optional) sanity check
trainable = sum(p.numel() for p in model_cls.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_cls.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


In [ ]:
train_df, eval_df = train_test_split(df, test_size=0.1, random_state=SEED)
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_dataset  = Dataset.from_pandas(eval_df.reset_index(drop=True))

# --- tokenize (keep multi-hot labels) ---
def tokenize(batch):
    enc = tok(batch["text"], truncation=True, padding="max_length", max_length=128)
    enc["labels"] = batch["labels"]  # multi-hot lists already in df["labels"]
    return enc

train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=train_dataset.column_names)
eval_dataset  = eval_dataset.map(tokenize,  batched=True, remove_columns=eval_dataset.column_names)

# Torch format
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format("torch",  columns=["input_ids", "attention_mask", "labels"])

In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred  # logits: (N, K), labels: (N, K)
    probs = 1 / (1 + np.exp(-logits))          # sigmoid
    preds = (probs >= 0.5).astype(int)         # threshold per-class; tuneable

    return {
        "f1_micro":  f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro":  f1_score(labels, preds, average="macro", zero_division=0),
        "precision_micro": precision_score(labels, preds, average="micro", zero_division=0),
        "recall_micro":    recall_score(labels, preds, average="micro", zero_division=0),
    }

In [ ]:

# -----------------------------
# Train
# -----------------------------
args = TrainingArguments(
    output_dir=OUT_DIR,
    learning_rate=2e-4,            
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=EPOCH,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="recall_micro",
    greater_is_better=True,
    seed=SEED,
    report_to="none",
)





In [ ]:
from transformers import EarlyStoppingCallback

callbacks = [EarlyStoppingCallback(
    early_stopping_patience=3,      # stop if no improvement for 3 evals
    early_stopping_threshold=0.0    # require strictly better metric; raise to ignore tiny noise
)]

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tok,
    compute_metrics=compute_metrics,
    callbacks=callbacks
)

trainer.train()

# -----------------------------
# Save: keep adapters separate
# -----------------------------
# Save only the NEW adapter (tox_cls) so you can stack it on top of the frozen DAPT later.
model.save_pretrained(OUT_DIR, selected_adapters=[NEW_ADAPTER_NAME])
# Optionally, also save the whole peft-wrapped model state (includes both adapters) for convenience:
# model.save_pretrained("./with_dapt_and_cls_adapters")

# Inference example:
# 1) Load base cls model
# 2) load dapt (frozen), then load cls adapter, set both active
# (same steps as above), then run tokenizer + model(**enc) -> logits

In [ ]:
# Test
!pip install better_profanity
import better_profanity

In [ ]:
from better_profanity import profanity
profanity.load_censor_words()
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

lyrics_path = '/content/drive/MyDrive/DALI/clean_explicit_comparison/lyrics_labeled.csv'
lyrics_df = pd.read_csv(lyrics_path).dropna(subset=['text'])
feature_df = lyrics_df['text']
true_pred = lyrics_df['label']
texts = feature_df.tolist()

In [ ]:
model.eval()
inputs = tok(texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
probs = torch.sigmoid(logits)

# Threshold to get binary predictions
threshold = 0.05
preds = (probs >= threshold).int()
preds_np = preds.cpu().numpy()                # convert to NumPy array
preds_multi = (preds_np.any(axis=1)).astype(int)  # 1 if any label positive else 0

In [ ]:
# Compare the results
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score
def print_results(true, pred):
  print("Precision: ", precision_score(true, pred))
  print("Recall: ", recall_score(true, pred))
  print("Accuracy: ", accuracy_score(true, pred))
  print("F1: ", f1_score(true, pred))

In [ ]:
# Only searching bad words
search_pred = lyrics_df['text'].astype(str).apply(profanity.contains_profanity)
search_pred = search_pred.to_numpy().astype(int)
print_results(true_pred,search_pred)

In [ ]:
# With finetuned language model
search_multi_pred = np.logical_or(preds_multi,search_pred).astype(int)
print_results(true_pred,search_multi_pred)